# 03 · Train and verify (B0 → B1 → B2 → B3s)
Fits the blending tables on training years and scores the held-out year (leave-one-year-out),
with a 95 % block-bootstrap confidence interval. Writes the scorecard, the slide-4 headline, weight maps
and the operational weights to `/kaggle/working/artifacts/`.

**Settings: Accelerator None. Internet ON (to clone the code).**
**Input:** right panel → *Add Input* → *Your Work* → the **01 ingest** notebook (its output holds `cache/`).

In [ ]:
# ── Setup (same in every notebook) ─────────────────────────────────────────────
# Kaggle: right panel → Settings → Internet ON.
import subprocess, sys, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gcsfs", "zarr>=2.18,<3"], check=True)

REPO = "/kaggle/working/repo"
BRANCH = "main"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    "https://github.com/shreyashsri79/081", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
sys.path.insert(0, REPO)
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "--short", "HEAD"]).decode().strip()
print("code commit:", COMMIT)

import numpy as np, pandas as pd, xarray as xr
from blend import config as C
np.random.seed(C.SEED)
print("xarray", xr.__version__)

In [ ]:
from blend.cache import find_cache, load_forecasts, load_truth
from blend.harmonise import prepare
from blend import verify as V
from blend import products as P
from blend.weights import dominant_model

SET = "S1"
VARIABLES = [C.T2M, C.WIND]
MODE = "loyo" if len(C.SETS[SET]["years"]) > 1 else "months"
CACHE = find_cache()
ART = "/kaggle/working/artifacts"
os.makedirs(f"{ART}/figures", exist_ok=True)
print("cache:", CACHE, sorted(os.listdir(CACHE)))

MODELS, YEARS = C.SETS[SET]["models"], C.SETS[SET]["years"]
forecasts = load_forecasts(CACHE, MODELS, YEARS)
truth = load_truth(CACHE)
print({m: dict(ds.sizes) for m, ds in forecasts.items()})

In [ ]:
cards, cells, choices = [], {}, {}
for var in VARIABLES:
    fc, obs = prepare(forecasts, truth, var)
    print(f"{var}: fc {dict(fc.sizes)}  cases with truth: {int(obs.notnull().any(['latitude','longitude']).sum())}")
    se, cell, chosen = V.run_folds(fc, obs, MODE)
    cards.append(V.scorecard(se, var, SET))
    cells[var], choices[var] = cell, chosen

card = pd.concat(cards, ignore_index=True)
card.to_csv(f"{ART}/scorecard_{SET}.csv", index=False)
cols = ["var", "lead_day", "rung", "rmse", "pct_vs_B0", "verdict_vs_B0", "pct_vs_B0bc", "verdict_vs_B0bc"]
card[card.lead_day.isin([1, 3, 5, 10])][cols].round(3)

In [ ]:
# ── The slide-4 lines: measured numbers only ──────────────────────────────────
lines = []
for var in VARIABLES:
    for ref in ["B0", "B0bc"]:
        lines.append(f"[vs {ref}] " + V.headline(card, var, lead_day=3, ref=ref))
lines.append(f"Set {SET}: {', '.join(MODELS)}; {MODE} over {YEARS}; 1.5° India box; 00 UTC; truth ERA5.")
open(f"{ART}/headline.txt", "w").write("\n".join(lines))
print("\n".join(lines))
print("\nB0 choices per fold:", choices)

In [ ]:
# ── Final fit on all years → operational weights + maps ─────────────────────
weights = {}
for var in VARIABLES:
    fc, obs = prepare(forecasts, truth, var)
    p = V.fit(fc, obs)
    weights[var] = xr.Dataset({"w_B2": p["w_B2"], "w_B3s": p["w_B3s"], "bias": p["bias"]})
    tag = var.replace("_", "")
    seasons = [str(x) for x in p["w_B3s"].season.values]
    SEAS = "JJAS" if "JJAS" in seasons else seasons[0]
    P.rmse_vs_lead(card, var, [f"m:{m}" for m in MODELS] + ["B0bc", "B1", "B2", "B3s"],
                   f"{ART}/figures/rmse_vs_lead_{tag}.png")
    P.weight_maps(p["w_B2"].sel(lead=3), f"{var} · B2 weights · Day 3", f"{ART}/figures/weights_B2_{tag}_d3.png")
    P.weight_maps(p["w_B3s"].sel(lead=3, season=SEAS), f"{var} · B3s weights · Day 3 · {SEAS}",
                  f"{ART}/figures/weights_B3s_{tag}_d3_{SEAS}.png")
    P.dominant_map(p["w_B3s"].sel(lead=3, season=SEAS), f"{var} · dominant model · Day 3 · {SEAS}",
                   f"{ART}/figures/dominant_{tag}_d3_{SEAS}.png")
    P.gain_map(cells[var], "B3s", "B0bc", 3, f"{var} · held-out RMSE change · Day 3",
               f"{ART}/figures/gain_B3s_vs_B0bc_{tag}_d3.png")
    ds = weights[var]
    ds.to_netcdf(f"{ART}/weights_{SET}_{tag}.nc")

import json, datetime
json.dump({"set": SET, "models": MODELS, "years": YEARS, "variables": VARIABLES, "mode": MODE,
           "code_commit": COMMIT, "alpha": C.ALPHA, "k_shrink": C.K_SHRINK, "smooth": C.SMOOTH,
           "n_boot": C.N_BOOT, "block_days": C.BLOCK_DAYS, "seed": C.SEED,
           "stores": {m: [u for u, _ in C.STORES[m]] for m in MODELS}, "truth": C.ERA5,
           "created_utc": datetime.datetime.utcnow().isoformat(timespec="seconds")},
          open(f"{ART}/manifest.json", "w"), indent=1)
print(sorted(os.listdir(ART)), sorted(os.listdir(f"{ART}/figures")))

In [ ]:
from IPython.display import Image, display
for f in sorted(os.listdir(f"{ART}/figures")):
    print(f); display(Image(f"{ART}/figures/{f}", width=700))